In [38]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("./files")
docs = loader.load()

In [39]:
docs[15].page_content

'GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS\n2.3 Outcome importance rating \nMembers of the WHO Steering Group, in consultation with the GDG and methodologist, developed a \nlist of treatment outcomes most relevant to the care of individuals with HTN. The GDG then rated each \noutcome on a scale from 1 to 9 and indicated whether it considered each outcome critical (rated 7–9), \nimportant (rated 4–6) or not important (rated 1–3) for decision-making. The mean scores are provided in \nAnnex 3. \n2.4 Reviews of evidence \nThe WHO Steering Group, with the participation of the GDG, determined the scope of the guideline \nand identified eleven questions in the format of population, intervention, comparison, and outcomes \n(PICO) to guide the search for systematic reviews (Annex 4). Eleven overviews of reviews informed the \nguideline development process. A systematic search was carried out in PubMed, Embase, The Cochrane \nLibrary, and Epistemonikos to identify exi

In [40]:
for doc in docs:
    doc.metadata["page"] = doc.metadata["page"] + 1

In [41]:
docs[0]

Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-02-11T15:30:29+00:00', 'moddate': '2022-11-30T11:03:18+00:00', 'trapped': '/False', 'source': 'files\\file1.pdf', 'total_pages': 61, 'page': 1, 'page_label': 'a'}, page_content='Guideline \nfor the\npharmacological \ntreatment of \nhypertension \nin adults')

In [42]:
import re

def clean_pdf_text(text: str) -> str:
    text = re.sub(r'-\n', '', text)

    text = re.sub(r'\n\s*\d{1,4}\s*\n', '\n', text)

    boilerplate_patterns = [
        # NICE (NG136)
        r'GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS'
        r'NICE guideline.*',
        r'© NICE \d{4}.*',
        r'Subject to Notice of rights.*',
        r'www\.nice\.org\.uk.*',
        r'National Institute for Health and Care Excellence.*',

        r'GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS',
        r'© World Health Organization \d{4}.*',
        r'World Health Organization;? ?\d{0,4}\.?$',
        r'ISBN 978-92-4-\d+-\d+.*',
        r'Some rights reserved.*',
        r'Creative Commons.*',
        r'creativecommons\.org.*',


        r'All rights reserved\.?$',
        r'^\s*(RECOMMENDATIONS|ANNEXES|REFERENCES|SPECIAL SETTINGS|'
        r'IMPLEMENTATION TOOLS|ACKNOWLEDGEMENTS|ACRONYMS AND ABBREVIATIONS|'
        r'EXECUTIVE SUMMARY|INTRODUCTION|METHOD FOR DEVELOPING THE GUIDELINE|'
        r'PUBLICATION, IMPLEMENTATION, EVALUATION AND RESEARCH GAPS)\s*$',
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


for doc in docs:
    doc.page_content = clean_pdf_text(doc.page_content)

print(f"{len(docs)} cleaned")
print("--- ------")
print(docs[16].page_content[:])
print("--- metadata ---")
print(docs[16].metadata)


455 cleaned
--- ------
2.5 Certainty of evidence and strength of recommendations 
The GDG rated the certainty of evidence and developed the recommendations using the GRADE 
(Grading of Recommendations Assessment, Development and Evaluation) approach (4). When making 
recommendations, GRADE defines the certainty of a body of evidence as “the extent of our confidence 
that the estimates of an effect are adequate to support a particular decision or recommendation” (5). 
Members of the GDG, with the help of the methodologist, developed evidence profiles to summarize 
relative and absolute estimates of effects, and an assessment of the certainty of the evidence. One 
evidence profile for each comparison within a PICO question was constructed, using the online 
Guideline Development Tool GRADEpro (https://gradepro.org). When there were systematic reviews 
addressing the relative effects in specific subpopulations, separate evidence profiles were constructed 
for each subpopulation.
According

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=600,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

In [44]:
len(chunks)

3919

In [45]:
chunks[0]

Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2022-02-11T15:30:29+00:00', 'moddate': '2022-11-30T11:03:18+00:00', 'trapped': '/False', 'source': 'files\\file1.pdf', 'total_pages': 61, 'page': 1, 'page_label': 'a'}, page_content='Guideline \nfor the\npharmacological \ntreatment of \nhypertension \nin adults')

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,        
    chunk_overlap=50,      
    separators=["\n\n", "\n", ". ", " ", ""], 
)


split_docs = splitter.split_documents(docs)

filename_to_docname = {
    "file1.pdf": "WHO Guideline - Pharmacological Treatment of Hypertension",
    "pdf2.pdf": "NICE NG136 - Hypertension in Adults",
}

all_chunks = []
for idx, chunk in enumerate(split_docs):
    source_file = chunk.metadata.get("source", "unknown")
    file_basename = source_file.split("/")[-1] 

    metadata ={
        "document_name": filename_to_docname.get(file_basename, file_basename),
        "page_number": chunk.metadata.get("page", 0),
        "chunk_id": f"{file_basename}_ch{idx:04d}",
        "source_url": source_file,
        # "text": chunk.page_content,
    }
    all_chunks.append(Document(page_content=chunk.page_content, metadata = metadata))


In [47]:
all_chunks[0]

Document(metadata={'document_name': 'files\\file1.pdf', 'page_number': 1, 'chunk_id': 'files\\file1.pdf_ch0000', 'source_url': 'files\\file1.pdf'}, page_content='Guideline \nfor the\npharmacological \ntreatment of \nhypertension \nin adults')

In [48]:
all_chunks[-1]

Document(metadata={'document_name': 'files\\pdf2.pdf', 'page_number': 52, 'chunk_id': 'files\\pdf2.pdf_ch5851', 'source_url': 'files\\pdf2.pdf'}, page_content="groups \n• the information on when to refer to the hypertension in pregnancy guideline was made \nclearer. \nThese recommendations are marked [2004, amended 2019] or [2011, amended 2019]. \nRecommendations marked [2004], [2006], [2008], [2009] and [2011] last had an \nevidence review in that year. In some cases minor changes have been made to the \nwording to bring the language and style up to date, without changing the meaning. \nRecommendations 1.2.12 and 1.4.24 (marked [2009]) were originally published in \nsection 1.4 of NICE's guideline on type 2 diabetes in adults, which was updated by this \nguideline. \nMinor changes since publication \nJune 2024: We corrected recommendation 1.4.26 to read 'antihypertensive' instead of \n'hypertensive'. \nOctober 2023: We corrected a link to an evidence review. \nJuly 2022: In recommenda

In [49]:
from langchain_huggingface import HuggingFaceEmbeddings

hf_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [50]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(all_chunks, hf_embeddings, persist_directory="chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=os.getenv("GROQ_API_KEY"))

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


In [77]:
def format_docs_with_metadata(docs):
    formatted_chunks = []
    for doc in docs:
        # Extract the specific metadata you want the LLM to see
        doc_name = doc.metadata.get('document_name', 'Unknown')
        page_num = doc.metadata.get('page_number', 'Unknown')
        chunk_id = doc.metadata.get('chunk_id', 'Unknown')
        
        # Format it so the LLM reads it as part of the text
        chunk_text = f"Source Document: {doc_name} | Page: {page_num} | chunk: {chunk_id} \nText:\n{doc.page_content}|"
        formatted_chunks.append(chunk_text)
        
    # Join all chunks together with double newlines
    return "\n\n---\n\n".join(formatted_chunks)

In [85]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """
            You are a helpful and polite assistant. 

            Follow these instructions strictly based on the type of user input:

            1. For casual greetings, pleasantries, or general small talk (e.g., "Hello", "How are you?"), respond naturally, politely, and conversationally.
            2. For any questions seeking specific information or facts, you must answer based ONLY on the provided context. 
            3. If the user asks for specific information and the context does not contain enough information to answer, you must reply exactly: "I don't know."
            4. When answering based on the context, you must include the source of your information at the very end of your response. Use this exact format:
            Source: [document_name], Page: [page_number], chunk_id: [chunk_id]
                - and if there is chunk repeated mention it only one time
            <context>
            {context}
            </context>
        """),
        ("human","{input}")
    ]
)

document_prompt = PromptTemplate(
    input_variables=["page_content", "document_name", "page_number", "chunk_id"],
    template="Source Document: {document_name} | Page: {page_number}| Chunk: {chunk_id} \nText:\n{page_content}"
)

stuff_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_prompt=document_prompt 
)
retriever_chain = create_retrieval_chain(retriever, stuff_chain)

In [86]:
retriever_chain.invoke({"input":"Hello"})

{'input': 'Hello',
 'context': [Document(id='c2401dc5-5d8a-41ea-a74b-e8403cb93340', metadata={'source_url': 'files\\file3.pdf', 'chunk_id': 'files\\file3.pdf_c2438', 'document_name': 'files\\file3.pdf', 'page_number': 180}, page_content='/gid00016/gid00029/gid00032/gid00046/gid00036/gid00047/gid00052/gid01142/gid00001/gid00028/gid00031/gid00048/gid00039/gid00047/gid00046/gid00001 /gid00028/gid00034/gid00032/gid00031/gid00001 /gid01088/gid01095/gid01748/gid00001 /gid00052/gid00032/gid00028/gid00045/gid00046/gid00001 /gid01175/gid01727/gid01176'),
  Document(id='3d5f2a8a-f443-4d75-b383-d45555210d90', metadata={'page_number': 186, 'chunk_id': 'files\\file3.pdf_c2604', 'source_url': 'files\\file3.pdf', 'document_name': 'files\\file3.pdf'}, page_content='/gid00016/gid00029/gid00032/gid00046/gid00036/gid00047/gid00052/gid01142/gid00001/gid00028/gid00031/gid00048/gid00039/gid00047/gid00046/gid00001 /gid00028/gid00034/gid00032/gid00031/gid00001 /gid01088/gid01095/gid01748/gid00001 /gid00052/gi

In [88]:
retriever_chain.invoke({"input":"What's Certainty of evidence and strength of recommendations? "})

{'input': "What's Certainty of evidence and strength of recommendations? ",
 'context': [Document(id='e38501c5-e4a0-4c82-bb5e-4e755799ae12', metadata={'document_name': 'files\\file1.pdf', 'chunk_id': 'files\\file1.pdf_c0026', 'page_number': 17, 'source_url': 'files\\file1.pdf'}, page_content='2.5 Certainty of evidence and strength of recommendations \nThe GDG rated the certainty of evidence and developed the recommendations using the GRADE \n(Grading of Recommendations Assessment, Development and Evaluation) approach (4). When making \nrecommendations, GRADE defines the certainty of a body of evidence as “the extent of our confidence \nthat the estimates of an effect are adequate to support a particular decision or recommendation” (5). \nMembers of the GDG, with the help of the methodologist, developed evidence profiles to summarize \nrelative and absolute estimates of effects, and an assessment of the certainty of the evidence. One \nevidence profile for each comparison within a PICO 

In [89]:
retriever_chain.invoke({"input":"what's hypertension in adults"})

{'input': "what's hypertension in adults",
 'context': [Document(id='efc2a8f7-3d6b-47ac-a621-6366f03b64b6', metadata={'source_url': 'files\\pdf2.pdf', 'chunk_id': 'files\\pdf2.pdf_c0136', 'document_name': 'files\\pdf2.pdf', 'page_number': 1}, page_content='Hypertension in adults: \ndiagnosis and management \nNICE guideline \nPublished: 28 August 2019 \nLast updated: 26 February 2026'),
  Document(id='f561e4bc-5193-49fb-b195-219357bcb3bb', metadata={'page_number': 1, 'chunk_id': 'files\\pdf2.pdf_c0136', 'document_name': 'files\\pdf2.pdf', 'source_url': 'files\\pdf2.pdf'}, page_content='Hypertension in adults: \ndiagnosis and management \nNICE guideline \nPublished: 28 August 2019 \nLast updated: 26 February 2026'),
  Document(id='5b4ae663-f027-4dbc-b4cf-f4eb5e607fd2', metadata={'document_name': 'files\\pdf2.pdf', 'chunk_id': 'files\\pdf2.pdf_c0136', 'page_number': 1, 'source_url': 'files\\pdf2.pdf'}, page_content='Hypertension in adults: \ndiagnosis and management \nNICE guideline \nPub

In [ ]:
while(True):
    question = input("ask your question?")
    if(question=="thanks"):
        break
    print(retriever_chain.invoke({"input":question})["answer"])
    print("\n--------------------------")

Hypertension in adults refers to high blood pressure in individuals aged 18 and above. It is a medical condition where the force of blood against the artery walls is consistently too high. This can lead to various health complications, such as heart disease, stroke, and kidney damage, if left untreated or poorly managed.

The guidelines for diagnosing and managing hypertension in adults are outlined in the provided NICE guideline document. However, more information is needed to provide a detailed explanation. For specific details, please refer to the document.

Source: files\pdf2.pdf, Page: 1, chunk_id: files\pdf2.pdf_c0136

--------------------------
Hypertension in adults, also known as high blood pressure, is a medical condition where the blood pressure in the arteries is consistently too high. This can lead to damage to the blood vessels, heart, and other organs if left untreated. It is a major risk factor for cardiovascular diseases, such as heart attacks, strokes, and kidney dise